# 05 — Deploy Endpoint + Monitoring (Model Monitor + CloudWatch)

This notebook demonstrates the “operability” parts of the system:

- Deploy latest **Approved** model from **Model Registry**
- Invoke the endpoint (show predictions)
- Enable **Data Capture**
- Create a **Model Monitor baseline** + monitoring schedule
- Create a basic **CloudWatch dashboard** (infrastructure monitoring)

This satisfies demo requirements:
- model registry
- endpoint invocation output
- monitoring reports
- infrastructure dashboards


In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import json
import time
import boto3
import sagemaker
import pandas as pd

from sagemaker.model import ModelPackage
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    DataCaptureConfig,
    DatasetFormat,
    CronExpressionGenerator,
    EndpointInput,
)

In [ ]:
# Load from previous notebook(s)
%store -r bucket
%store -r region
%store -r RUN_ID
%store -r MODEL_PACKAGE_GROUP

print("Bucket:", bucket)
print("Region:", region)
print("RUN_ID:", RUN_ID)
print("MODEL_PACKAGE_GROUP:", MODEL_PACKAGE_GROUP)

In [ ]:
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
sm = boto3.client("sagemaker", region_name=region)

print("Role:", role)

In [ ]:
# Get latest APPROVED model package from the Model Package Group
resp = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    ModelApprovalStatus="Approved",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)

if not resp["ModelPackageSummaryList"]:
    raise RuntimeError(
        "No APPROVED model packages found. "
        "Run Notebook 04, or approve a model in the Model Registry."
    )

model_package_arn = resp["ModelPackageSummaryList"][0]["ModelPackageArn"]
print("ModelPackageArn:", model_package_arn)

In [ ]:
# Deploy endpoint with data capture enabled
endpoint_name = f"buoycast-endpoint-{RUN_ID}"

data_capture_s3 = f"s3://{bucket}/buoycast/datacapture/{RUN_ID}/"
data_capture = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=data_capture_s3,
)

model = ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=sess,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=endpoint_name,
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
    data_capture_config=data_capture,
)

print("Deployed endpoint:", endpoint_name)
print("Data capture S3:", data_capture_s3)

In [ ]:
# Invoke endpoint using the pipeline-produced inference sample
sample_s3_uri = f"s3://{bucket}/buoycast/artifacts/{RUN_ID}/data/baseline/inference_sample.csv"
sample_df = pd.read_csv(sample_s3_uri)

payload = sample_df.to_csv(index=False, header=False)
pred = predictor.predict(payload)

print("Sample rows:", len(sample_df))
pred

In [ ]:
# Model Monitor baseline + schedule (Data Quality monitor)

monitor_output_s3 = f"s3://{bucket}/buoycast/model-monitor/{RUN_ID}/"

baseline_s3_uri = f"s3://{bucket}/buoycast/artifacts/{RUN_ID}/data/baseline/baseline.csv"

monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sess,
)

# 1) Suggest baseline (generates statistics + constraints)
monitor.suggest_baseline(
    baseline_dataset=baseline_s3_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"{monitor_output_s3}baseline/",
    wait=True,
)

print("Baseline statistics:", monitor.baseline_statistics())
print("Baseline constraints:", monitor.baseline_constraints())

In [ ]:
# 2) Create a monitoring schedule on the endpoint
schedule_name = f"buoycast-data-quality-{RUN_ID}"

monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name,
    endpoint_input=EndpointInput(
        endpoint_name=endpoint_name,
        destination="/opt/ml/processing/input",
    ),
    output_s3_uri=f"{monitor_output_s3}reports/",
    statistics=monitor.baseline_statistics(),
    constraints=monitor.baseline_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)

print("Created monitoring schedule:", schedule_name)
print("Monitor outputs:", monitor_output_s3)

In [ ]:
# CloudWatch dashboard (endpoint metrics)
cw = boto3.client("cloudwatch", region_name=region)

dashboard_name = f"BuoyCast-{RUN_ID}"

dashboard = {
    "widgets": [
        {
            "type": "metric",
            "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", endpoint_name, "VariantName", "AllTraffic"],
                    [".", "Invocation4XXErrors", ".", ".", ".", "."],
                    [".", "Invocation5XXErrors", ".", ".", ".", "."],
                ],
                "region": region,
                "stat": "Sum",
                "period": 300,
                "title": "Endpoint Invocations + Errors"
            }
        },
        {
            "type": "metric",
            "x": 0, "y": 6, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "ModelLatency", "EndpointName", endpoint_name, "VariantName", "AllTraffic"],
                    [".", "OverheadLatency", ".", ".", ".", "."],
                ],
                "region": region,
                "stat": "Average",
                "period": 300,
                "title": "Endpoint Latency (ms)"
            }
        },
    ]
}

cw.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard),
)

print("Created/updated dashboard:", dashboard_name)

In [ ]:
# Save for cleanup notebook
%store endpoint_name
%store schedule_name
%store dashboard_name
%store data_capture_s3
%store monitor_output_s3